# UNE Unwrapped — Backfill AI on Colab GPU

Procesa el histórico de mensajes (`telegram_messages.db`) con el pipeline de IA del proyecto, aprovechando la GPU gratuita de Colab (T4 ≈ 15 GB VRAM).

**Antes de empezar:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **GPU** (T4 está bien).
2. Clona el repo público (no requiere credenciales).
3. Si no quieres tocar el repo, este notebook descarga y devuelve un `.zip` con la BD y los JSONs procesados — listo para subir a tu repo.

**Tiempos esperados (T4):**
- Setup + warm models: ~2-3 min
- 5,000 mensajes/año: ~30-60 s
- Histórico completo (~60k): ~6-12 min

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Clonar el repo y entrar al directorio

Cambia `REPO_URL` si tienes un fork. Por defecto usa la rama `master`.
Si tu repo es privado, primero ejecuta `from google.colab import userdata; token = userdata.get('GH_TOKEN')` y usa `https://{token}@github.com/...`.

In [ ]:
import os, subprocess, shutil

REPO_URL = 'https://github.com/eduar-hte/une-unwrapped-habana.git'  # cambia por tu fork si aplica
REPO_DIR = '/content/une-unwrapped-habana'
BRANCH   = 'master'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR
!ls -la

## 3. Instalar dependencias

Instalamos en el entorno actual de Colab (que ya trae torch CUDA). Solo añadimos `transformers`, `optimum`, `huggingface-hub`, `psutil`, `telethon`, `python-dotenv`, `pytz`, `tzdata`. **No** necesitamos `onnxruntime` ya que la GPU usa el backend PyTorch.

In [ ]:
!pip install -q 'transformers>=4.46,<5.0' 'huggingface-hub>=0.26' 'optimum>=1.24,<2.0' 'telethon==1.42.0' 'python-dotenv==1.2.1' 'pytz==2025.2' 'tzdata==2025.3' 'pyaes==1.6.1' 'pyasn1==0.6.1' 'rsa==4.9.1' 'psutil>=6.0,<8.0'

## 4. Detectar backend y warm-up de modelos

Con GPU detectada, los modelos se cargan en CUDA con fp16 (sin necesidad de cuantización ONNX). El warm-up descarga ~700 MB la primera vez, luego se sirve del caché de Colab.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s %(message)s')

# Force GPU backend explicitly (the default auto-detect would also pick it up).
import os
os.environ['UNE_AI_BACKEND'] = 'gpu'

from core.ai.models import detect_backend, gpu_info, warm_models
print('Backend:', detect_backend())
print('GPU info:', gpu_info())
warm_models()

## 5. (Opcional) Validar accuracy con el eval set

In [ ]:
!python -m core.ai.eval

## 6. Backfill por año (o todos los años)

Cambia `YEARS` si quieres procesar solo algunos. Por defecto procesa el rango completo presente en la BD del repo (2022 → 2026).

El parámetro `BATCH_SIZE` controla cuántos mensajes se procesan antes de cada commit a SQLite. En GPU `512-1024` funcionan bien; en T4 limita el ritmo a ~5000-15000 msgs/min.

In [ ]:
import time
from core.ai.processor import process_pending_ai_analysis
from core import analyze_data

# === CONFIG ===
YEARS = [2022, 2023, 2024, 2025, 2026]   # set [2022] for a smoke test
BATCH_SIZE = 512
MAX_MESSAGES_PER_YEAR = None             # int para tope, None = sin tope
# ==============

# Provide channel_username for the link builder used by the analyzer.
os.environ.setdefault('CHANNEL_USERNAME', 'EmpresaElectricaDeLaHabana')

global_start = time.time()
for year in YEARS:
    print(f'\n========== Year {year} ==========')
    t0 = time.time()
    stats = process_pending_ai_analysis(
        batch_size=BATCH_SIZE,
        max_messages=MAX_MESSAGES_PER_YEAR,
        year_filter=year,
    )
    print(f'AI stats {year}: {stats}')
    print(f'Re-running analyzer for {year}...')
    analyze_data(year)
    print(f'Year {year} done in {time.time()-t0:.1f}s')

print(f'\nTotal elapsed: {(time.time()-global_start)/60:.1f} min')

## 7. Empaquetar resultados

Empaqueta `telegram_messages.db` y los JSONs anuales en un zip listo para descargar y subir al repo (commit manual).

In [ ]:
import shutil, time, os
from pathlib import Path

OUT_DIR = Path('/content/une_backfill_output')
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True)

shutil.copy('telegram_messages.db', OUT_DIR / 'telegram_messages.db')
json_dir = OUT_DIR / 'app' / 'public' / 'data'
json_dir.mkdir(parents=True, exist_ok=True)
for js in Path('app/public/data').glob('analysis_data_*.json'):
    shutil.copy(js, json_dir / js.name)

ts = time.strftime('%Y%m%d_%H%M%S')
zip_base = f'/content/une_backfill_{ts}'
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=OUT_DIR)
print(f'Bundle ready: {zip_path}')
print(f'Size: {os.path.getsize(zip_path)/1024/1024:.1f} MB')

## 8. Descargar zip a tu máquina

In [ ]:
from google.colab import files
files.download(zip_path)

## 9. (Opcional) Push directo al repo

Si prefieres que Colab haga commit + push directamente, descomenta el bloque y guarda tu token en `Secrets` de Colab como `GH_TOKEN` (necesita permiso `contents:write`).

Ten en cuenta: si el repo tiene `branch protection` o el push es a `master`, considera usar una rama `colab-backfill` y abrir PR.

```python
# from google.colab import userdata
# token = userdata.get('GH_TOKEN')
# !git config user.name 'colab-backfill'
# !git config user.email 'colab@local'
# !git remote set-url origin https://{token}@github.com/eduar-hte/une-unwrapped-habana.git
# !git checkout -b colab-backfill-$(date +%s)
# !git add telegram_messages.db app/public/data/analysis_data_*.json
# !git commit -m 'AI backfill from Colab GPU'
# !git push -u origin HEAD
```

## Tips & troubleshooting

- **Si Colab desconecta antes de terminar**: el procesamiento es resumible — la BD guarda `model_version` por mensaje, así que al re-ejecutar el notebook saltará lo ya procesado.
- **OOM en GPU**: baja `BATCH_SIZE` a 128 o 64.
- **Quieres más velocidad (Pro/A100)**: sube `BATCH_SIZE` a 1024 o 2048.
- **Precision check**: la celda 5 corre `core.ai.eval` que valida 15 casos representativos contra los esperados. Top-1 accuracy debería ser >95%.